In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import GradientBoostingRegressor


In [ ]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score


In [ ]:
hourly = pd.read_csv('../01_Data/processed/model_ready_hourly.csv', parse_dates=['date_time'])


In [ ]:
hourly.head()


In [ ]:
features = ["hour","dayofweek","month","is_weekend","lag_1","lag_24","lag_168","rolling_24","rolling_168"]


## Train test split

In [ ]:
cut = int(len(hourly) * 0.8)


In [ ]:
train = hourly.iloc[:cut]
test = hourly.iloc[cut:]


In [ ]:
X_train = train[features]
y_train = train["energy_kwh"]


In [ ]:
X_test = test[features]
y_test = test["energy_kwh"]


## Weekly baseline

In [ ]:
baseline = test["lag_168"]


In [ ]:
base_mae = mean_absolute_error(y_test, baseline)


In [ ]:
print("Baseline MAE", round(base_mae, 3))


## Random Forest

In [ ]:
rf = RandomForestRegressor(n_estimators=100, max_depth=12, random_state=42, n_jobs=-1)


In [ ]:
rf.fit(X_train, y_train)


In [ ]:
pred_rf = rf.predict(X_test)


In [ ]:
rf_mae = mean_absolute_error(y_test, pred_rf)


In [ ]:
rf_rmse = mean_squared_error(y_test, pred_rf) ** 0.5


In [ ]:
rf_r2 = r2_score(y_test, pred_rf)


In [ ]:
print("RF MAE", round(rf_mae, 3))
print("RF RMSE", round(rf_rmse, 3))
print("RF R2", round(rf_r2, 3))


## Gradient Boosting

In [ ]:
gb = GradientBoostingRegressor(n_estimators=100, max_depth=2, learning_rate=0.04, random_state=42)


In [ ]:
gb.fit(X_train, y_train)


In [ ]:
pred_gb = gb.predict(X_test)


In [ ]:
gb_mae = mean_absolute_error(y_test, pred_gb)


In [ ]:
gb_rmse = mean_squared_error(y_test, pred_gb) ** 0.5


In [ ]:
gb_r2 = r2_score(y_test, pred_gb)


In [ ]:
print("GB MAE", round(gb_mae, 3))
print("GB RMSE", round(gb_rmse, 3))
print("GB R2", round(gb_r2, 3))


## MAE improvement

In [ ]:
improvement = (base_mae - gb_mae) / base_mae * 100


In [ ]:
print("MAE improvement", round(improvement, 2), "%")


## Feature importance

In [ ]:
importance = pd.Series(gb.feature_importances_, index=features)


In [ ]:
importance = importance.sort_values(ascending=False)


In [ ]:
importance


In [ ]:
importance.plot(kind="bar", figsize=(9,4), title="Feature Importance")
plt.tight_layout()
plt.show()


In [ ]:
plt.savefig('../04_Visuals/ml/07_feature_importance.png', dpi=150)
plt.show()


## Forecast check

In [ ]:
plt.figure(figsize=(10,4))
plt.plot(test["date_time"].tail(168), y_test.tail(168), label="Actual")
plt.plot(test["date_time"].tail(168), pred_gb[-168:], label="Gradient Boosting")
plt.legend()
plt.ylabel("kWh")
plt.title("Actual vs Forecast")
plt.show()


In [ ]:
plt.savefig('../04_Visuals/forecasting/10_forecast_actual_vs_pred.png', dpi=150)
plt.show()


## High use days

In [ ]:
daily = pd.read_csv('../01_Data/processed/daily_energy.csv', parse_dates=['date'])


In [ ]:
daily["lag_1"] = daily["daily_kwh"].shift(1)


In [ ]:
daily["lag_7"] = daily["daily_kwh"].shift(7)


In [ ]:
daily["rolling_7"] = daily["daily_kwh"].shift(1).rolling(7).mean()


In [ ]:
daily = daily.dropna()


In [ ]:
cut = int(len(daily) * 0.8)
train = daily.iloc[:cut]
test = daily.iloc[cut:]


In [ ]:
limit = train["daily_kwh"].quantile(0.75)


In [ ]:
train["high_use"] = (train["daily_kwh"] > limit).astype(int)
test["high_use"] = (test["daily_kwh"] > limit).astype(int)


## Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier


In [ ]:
from sklearn.metrics import accuracy_score


In [ ]:
day_features = ["dayofweek","month","is_weekend","lag_1","lag_7","rolling_7"]


In [ ]:
# using the same train and test data


In [ ]:
train[["daily_kwh","high_use"]].head()


In [ ]:
dt = DecisionTreeClassifier(max_depth=3, random_state=42)


In [ ]:
dt.fit(train[day_features], train["high_use"])


In [ ]:
pred_dt = dt.predict(test[day_features])


In [ ]:
print("Decision Tree accuracy", round(accuracy_score(test["high_use"], pred_dt), 3))


## Random Forest alerts

In [ ]:
from sklearn.ensemble import RandomForestClassifier


In [ ]:
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score


In [ ]:
rf_alert = RandomForestClassifier(n_estimators=100, max_depth=5, class_weight="balanced", random_state=42, n_jobs=-1)


In [ ]:
rf_alert.fit(train[day_features], train["high_use"])


In [ ]:
pred_alert = rf_alert.predict(test[day_features])


In [ ]:
print("Accuracy", round(accuracy_score(test["high_use"], pred_alert), 3))


In [ ]:
print("Precision", round(precision_score(test["high_use"], pred_alert), 3))


In [ ]:
print("Recall", round(recall_score(test["high_use"], pred_alert), 3))


In [ ]:
print("F1", round(f1_score(test["high_use"], pred_alert), 3))


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
ConfusionMatrixDisplay.from_predictions(test["high_use"], pred_alert)
plt.title("High Use Day Alerts")
plt.show()


In [ ]:
plt.savefig('../04_Visuals/ml/09_confusion_matrix.png', dpi=150)
plt.show()
